In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Dell User\\Desktop\\DeepLearm\\Deep_Learning\\research'

In [3]:
os.chdir(r"C:\Users\Dell User\Desktop\DeepLearm\Deep_Learning")

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [5]:
# configuration manager
from src.constants import *
from src.utils.common import read_yaml,create_directories
import tensorflow as tf 

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        training_data = os.path.join(
            self.config.data_ingestion.unzip_dir,
            "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
        )

        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [7]:
import os 
import tensorflow as tf 
from zipfile import ZipFile
import urllib.request as request



class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
    def get_base_model(self):
        self.model=tf.keras.models.load_model(self.config.updated_base_model_path)
  
    def train_valid_generator(self):
        from tensorflow.keras.applications.densenet import preprocess_input

        datagenerator_kwargs = dict(
            preprocessing_function=preprocess_input, # <-- CRITICAL: Align input distribution
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=10,
                horizontal_flip=True,
                vertical_flip=True, 
                width_shift_range=0.2,
                height_shift_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

        callbacks = [
            EarlyStopping(
                monitor="val_accuracy",
                patience=15,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="val_accuracy",
                factor=0.5,
                patience=7,
                min_lr=1e-7
            ),
            ModelCheckpoint(
                filepath=str(self.config.trained_model_path),  # saves to your config path
                monitor="val_accuracy",
                save_best_only=True
            )
        ]
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callbacks
          
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [ ]:
# import os
# import tensorflow as tf
# from pathlib import Path
# from src.entity.config_entity import TrainingConfig


# class Training:
#     def __init__(self, config: TrainingConfig):
#         self.config = config

#     def get_base_model(self):
#         self.model = tf.keras.models.load_model(
#             self.config.updated_base_model_path
#         )
#         self.model.compile(
#             optimizer=tf.keras.optimizers.Adam(
#                 learning_rate=self.config.params_learning_rate
#             ),
#             loss=tf.keras.losses.CategoricalCrossentropy(),
#             metrics=["accuracy"]
#         )

#     def train_valid_generator(self):
#         AUTOTUNE = tf.data.AUTOTUNE
#         image_size = tuple(self.config.params_image_size[:-1])  # (224, 224)

#         # rescaling layer — replaces 1./255
#         rescale = tf.keras.layers.Rescaling(1.0 / 255)

#         train_ds = tf.keras.utils.image_dataset_from_directory(
#             self.config.training_data,
#             validation_split=0.20,
#             subset="training",
#             seed=42,
#             image_size=image_size,
#             batch_size=self.config.params_batch_size,
#             label_mode="categorical"  # ← one-hot labels for CategoricalCrossentropy
#         )

#         val_ds = tf.keras.utils.image_dataset_from_directory(
#             self.config.training_data,
#             validation_split=0.20,
#             subset="validation",
#             seed=42,
#             image_size=image_size,
#             batch_size=self.config.params_batch_size,
#             label_mode="categorical"
#         )

#         if self.config.params_is_augmentation:
#             augmentation = tf.keras.Sequential([
#                 tf.keras.layers.RandomFlip("horizontal"),
#                 tf.keras.layers.RandomRotation(0.2),
#                 tf.keras.layers.RandomZoom(0.2),
#                 tf.keras.layers.RandomTranslation(0.2, 0.2),
#             ])
#             # apply augmentation only to training set
#             train_ds = train_ds.map(
#                 lambda x, y: (augmentation(x, training=True), y),
#                 num_parallel_calls=AUTOTUNE
#             )

#         # rescale both
#         train_ds = train_ds.map(
#             lambda x, y: (rescale(x), y),
#             num_parallel_calls=AUTOTUNE
#         )
#         val_ds = val_ds.map(
#             lambda x, y: (rescale(x), y),
#             num_parallel_calls=AUTOTUNE
#         )

#         # cache + prefetch — keeps GPU fully fed
#         self.train_generator = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
#         self.valid_generator = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

#     @staticmethod
#     def save_model(path: Path, model: tf.keras.Model):
#         model.save(path)

#     def train(self):
#         from tensorflow.keras.callbacks import (
#             EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
#         )

#         callbacks = [
#             EarlyStopping(
#                 monitor="val_accuracy",
#                 patience=15,
#                 restore_best_weights=True
#             ),
#             ReduceLROnPlateau(
#                 monitor="val_accuracy",
#                 factor=0.5,
#                 patience=7,
#                 min_lr=1e-7
#             ),
#             ModelCheckpoint(
#                 filepath=str(self.config.trained_model_path),
#                 monitor="val_accuracy",
#                 save_best_only=True
#             )
#         ]

#         self.model.fit(
#             self.train_generator,
#             epochs=self.config.params_epochs,
#             validation_data=self.valid_generator,
#             callbacks=callbacks
#         )

#         self.save_model(
#             path=self.config.trained_model_path,
#             model=self.model
#         )

In [8]:

try:
    config=ConfigurationManager()
    training_config =config.get_training_config()
    training = Training(training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e

[2026-06-28 17:51:57,217: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 17:51:57,219: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 17:51:57,220: INFO: common: created directory at: artifacts]
[2026-06-28 17:51:57,221: INFO: common: created directory at: artifacts\training]
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.
Epoch 1/100
622/622 [==============================] - 173s 260ms/step - loss: 0.8969 - accuracy: 0.6508 - val_loss: 0.9171 - val_accuracy: 0.6694 - lr: 1.0000e-04
Epoch 2/100
622/622 [==============================] - 153s 246ms/step - loss: 0.5197 - accuracy: 0.8033 - val_loss: 0.9088 - val_accuracy: 0.6665 - lr: 1.0000e-04
Epoch 3/100
622/622 [==============================] - 156s 250ms/step - loss: 0.4318 - accuracy: 0.8407 - val_loss: 0.9167 - val_accuracy: 0.6339 - lr: 1.0000e-04
Epoch 4/100
622/622 [==============================] - 153s 245ms/step - loss: 0.3831 - accura

KeyboardInterrupt: 

In [10]:
import tensorflow as tf
print("Is GPU available? ", tf.config.list_physical_devices('GPU'))


Is GPU available?  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [13]:
import sys
import tensorflow as tf

print("Python path:", sys.executable)
print("TensorFlow version:", tf.__version__)
print("Is GPU available? ", tf.config.list_physical_devices('GPU'))


Python path: c:\Users\Dell User\Desktop\DeepLearm\Deep_Learning\.venv\Scripts\python.exe
TensorFlow version: 2.10.0
Is GPU available?  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
